#**Take-Home-Exercise2-Joey Zhu**

###**0.Data Setup**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "dbfs:/Volumes/gr5069/raw/f1_data"

pit_stops    = spark.read.csv(f"{BASE}/pit_stops.csv",    header=True, inferSchema=True)
races        = spark.read.csv(f"{BASE}/races.csv",         header=True, inferSchema=True)
drivers      = spark.read.csv(f"{BASE}/drivers.csv",       header=True, inferSchema=True)
results      = spark.read.csv(f"{BASE}/results.csv",       header=True, inferSchema=True)
constructors = spark.read.csv(f"{BASE}/constructors.csv",  header=True, inferSchema=True)
status       = spark.read.csv(f"{BASE}/status.csv",        header=True, inferSchema=True)

###**1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.**

**Q1 Logic**

The pit_stops table has one row per individual pit stop, containing raceId, driverId, and milliseconds (how long that stop took). A driver can pit multiple times in a race, so I need to group by (raceId, driverId) and compute the average, minimum (fastest), and maximum (slowest) duration. I then join with races and drivers to replace IDs with readable names.

In [0]:
q1 = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.avg("milliseconds").alias("avg_pit_ms"),
        F.min("milliseconds").alias("fastest_pit_ms"),
        F.max("milliseconds").alias("slowest_pit_ms"),
    )
    .join(races.select("raceId", "name", "year"), on="raceId")
    .join(drivers.select("driverId", "surname", "forename"), on="driverId")
    .orderBy("raceId", "avg_pit_ms")
)

display(q1)

**Explannation:**

groupBy("raceId", "driverId") partitions the data so all pit stops for one driver in one race are grouped together. Inside .agg(...), F.avg, F.min, and F.max are applied to the milliseconds column — each computes a single summary value per group. .alias(...) renames the output columns. The first .join attaches the race name and year from the races table using the shared raceId key. The second .join attaches the driver's name from drivers using driverId. .orderBy("raceId", "avg_pit_ms") sorts the result by race, then by fastest average pit time within that race.

**Alternative approach:**
You could register the DataFrames as temporary views and write this as a SQL query using GROUP BY and MIN/MAX/AVG, which some find more readable:

In [0]:
pit_stops.createOrReplaceTempView("pit_stops")
spark.sql("""
  SELECT raceId, driverId, AVG(milliseconds) AS avg_pit_ms,
         MIN(milliseconds) AS fastest_pit_ms, MAX(milliseconds) AS slowest_pit_ms
  FROM pit_stops GROUP BY raceId, driverId
""")

### 2. Rank order by finishing position the average time spent at the pit stop in each race.


**Q2 Logic**

Starting from Question 1's per-driver averages, I want to assign each driver a rank within their race — rank 1 meaning the fastest average pit time in that race. This requires a window function that resets the ranking for each race independently. I partition the window by raceId and order by avg_pit_ms ascending.

In [0]:
win_race = Window.partitionBy("raceId").orderBy("avg_pit_ms")

q2 = (
    q1
    .withColumn("pit_rank", F.rank().over(win_race))
    .select(
        "raceId", "name", "year",
        "forename", "surname",
        "avg_pit_ms", "fastest_pit_ms", "slowest_pit_ms",
        "pit_rank"
    )
    .orderBy("raceId", "pit_rank")
)

display(q2)

**Explanation:**

Window.partitionBy("raceId") creates a separate computation window for every race, so ranks are relative within a race and do not bleed across races. .orderBy("avg_pit_ms") sorts within each window so the driver with the lowest average pit time gets rank 1. F.rank().over(win_race) computes the rank using that window — rank() gives tied drivers the same number and then skips the next rank (e.g. 1, 1, 3). .withColumn("pit_rank", ...) adds this as a new column without removing existing ones. The final .select picks only the meaningful columns.



**Alternative:**

 Use F.dense_rank() instead of F.rank() to avoid gaps in the ranking sequence (ties become 1, 1, 2 instead of 1, 1, 3). Use F.row_number() if you need a strict unique ordering with no ties.

### 3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

**Q3 Logic:**

The drivers table has a code column (e.g. HAM, VET) that is sometimes null or the literal string \N (the CSV null sentinel Databricks writes for missing values). The official F1 rule for generating a driver code is the first three letters of the surname, uppercased — for example, Alonso → ALO. I apply this rule only where code is missing, preserving existing codes for all others.

In [0]:
# Step 1: Convert "\N" CSV sentinels to proper nulls
drivers_clean = drivers.withColumn(
    "code",
    F.when(F.col("code").isin("\\N", ""), None).otherwise(F.col("code"))
)

# Step 2: Fill nulls with first 3 uppercase letters of surname
drivers_filled = drivers_clean.withColumn(
    "code",
    F.when(
        F.col("code").isNull(),
        F.upper(F.substring("surname", 1, 3))
    ).otherwise(F.col("code"))
)

display(
    drivers_filled
    .select("driverId", "forename", "surname", "code")
    .orderBy("surname")
)

**Explanation**

Step 1 uses F.when(...).otherwise(...) — a conditional expression — to replace the literal string "\\N" (and empty strings) with None, which Spark treats as a true SQL null. Without this step, isNull() would not catch those rows. Step 2 applies the same conditional pattern: F.col("code").isNull() identifies rows that still have no code, and F.upper(F.substring("surname", 1, 3)) derives the code — F.substring(col, 1, 3) extracts characters starting at position 1 (1-indexed in Spark) for a length of 3, and F.upper uppercases the result. Rows where code is already populated pass through .otherwise(F.col("code")) unchanged.

**Alternative:**

 For drivers whose surnames are shorter than 3 characters, substring silently returns fewer letters. If you need strict 3-character codes padded to length, wrap with F.rpad(F.upper(F.substring("surname",1,3)), 3, "X").

### 4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

**Q4 Logic**

Definition of "Age": Age at race time, measured in whole completed years as of the race date. The formula is floor((race_date − date_of_birth) / 365.25). I use 365.25 to account for leap years. This matches the conventional social meaning of age — you do not turn 31 until your full 31st year is complete.
Steps: join results → races (to get the race date) → drivers (to get date of birth), compute the Age column, then use a window function to broadcast the min and max age within each race and filter for the matching rows.

In [0]:
race_driver = (
    results
    .select("raceId", "driverId")
    .join(races.select("raceId", "date"), on="raceId")
    .join(drivers.select("driverId", "forename", "surname", "dob"), on="driverId")
    .withColumn("race_date",  F.to_date("date"))
    .withColumn("birth_date", F.to_date("dob"))
    .withColumn(
        "Age",
        F.floor(F.datediff("race_date", "birth_date") / 365.25).cast("int")
    )
)

win_age = Window.partitionBy("raceId")

age_result = (
    race_driver
    .withColumn("min_age", F.min("Age").over(win_age))
    .withColumn("max_age", F.max("Age").over(win_age))
    .filter(
        (F.col("Age") == F.col("min_age")) | (F.col("Age") == F.col("max_age"))
    )
    .withColumn(
        "category",
        F.when(F.col("Age") == F.col("min_age"), "Youngest")
         .when(F.col("Age") == F.col("max_age"), "Oldest")
    )
    .join(races.select("raceId", "name", "year"), on="raceId")
    .select("raceId", "name", "year", "forename", "surname", "Age", "category")
    .orderBy("raceId", "category")
)

display(age_result)

**Explanation**

F.to_date("date") and F.to_date("dob") parse the string columns into Spark DateType values so date arithmetic is possible. F.datediff("race_date", "birth_date") returns the number of calendar days between the two dates. Dividing by 365.25 converts days to fractional years, and F.floor(...) truncates to whole completed years — the standard definition of age. Window.partitionBy("raceId") with no .orderBy() creates a window over all rows in a race, so F.min("Age").over(win_age) and F.max("Age").over(win_age) broadcast the race's youngest and oldest ages as new columns on every row. The .filter(...) then keeps only the rows where a driver's age matches one of those extremes. The .when(...) labels each row "Youngest" or "Oldest".

**Alternative:**

Use F.months_between("race_date", "birth_date") / 12 instead of datediff / 365.25 for an age calculation that is more precise around month boundaries.

### 5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

**Logic**

"At any given race" means: up to and including that race, how many wins, 2nd places, and 3rd places has each driver accumulated? This is a running (cumulative) total over time. I use a window partitioned by driver and ordered by race date with an unbounded preceding frame, so each row's count includes all prior races plus the current one. I use positionOrder (always numeric) rather than position (which can contain strings like "R" for retired).

In [0]:
results_dated = (
    results
    .join(races.select("raceId", "date", "name", "year"), on="raceId")
    .join(drivers.select("driverId", "surname", "forename"), on="driverId")
    .withColumn("race_date", F.to_date("date"))
    .withColumn("is_win", (F.col("positionOrder") == 1).cast("int"))
    .withColumn("is_2nd", (F.col("positionOrder") == 2).cast("int"))
    .withColumn("is_3rd", (F.col("positionOrder") == 3).cast("int"))
)

win_cum = (
    Window
    .partitionBy("driverId")
    .orderBy("race_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

q5 = (
    results_dated
    .withColumn("total_wins", F.sum("is_win").over(win_cum))
    .withColumn("total_2nd",  F.sum("is_2nd").over(win_cum))
    .withColumn("total_3rd",  F.sum("is_3rd").over(win_cum))
    .withColumn(
        "total_podiums",
        F.col("total_wins") + F.col("total_2nd") + F.col("total_3rd")
    )
    .select(
        "raceId", "name", "year", "race_date",
        "forename", "surname", "positionOrder",
        "total_wins", "total_2nd", "total_3rd", "total_podiums"
    )
    .orderBy("race_date", "surname")
)

display(q5)

**Explanation:**

(F.col("positionOrder") == 1).cast("int") turns a boolean comparison into a 1 (finished 1st) or 0 (did not), making it summable. The same logic applies for is_2nd and is_3rd. Window.partitionBy("driverId") isolates each driver's history — one driver's count never bleeds into another's. .orderBy("race_date") ensures races are processed chronologically. .rowsBetween(Window.unboundedPreceding, Window.currentRow) defines the frame: from the very first row in the partition through the current row, giving a running total. F.sum("is_win").over(win_cum) applies that sum to each row, so each race entry shows the driver's cumulative wins as of that race.

**Alternative:**

A self-join on race_date_A <= race_date_B followed by a groupBy achieves the same result, but the window approach is far more efficient in Spark because it avoids a potentially massive Cartesian-style join.

### 6. Continue exploring the data by answering your own question.
### 

**Logic:**

Join results → status (to get the failure description) → races (for year) → constructors (for team name). Filter for status descriptions containing mechanical failure keywords (engine, gearbox, hydraulics, etc.). Group by constructor and decade, count the rows.

In [0]:
mechanical_keywords = [
    "engine", "gearbox", "hydraulic", "suspension",
    "brakes", "electrical", "fuel", "oil", "turbo", "mechanical"
]

# Build a combined OR filter from all keywords
keyword_filter = F.lit(False)
for kw in mechanical_keywords:
    keyword_filter = keyword_filter | F.lower(F.col("status")).contains(kw)

q6 = (
    results
    .join(status.select("statusId", "status"),            on="statusId")
    .join(races.select("raceId", "year"),                 on="raceId")
    .join(constructors.select("constructorId", "name"),   on="constructorId")
    .filter(keyword_filter)
    .withColumn("decade", (F.floor(F.col("year") / 10) * 10).cast("int"))
    .groupBy("name", "decade")
    .agg(F.count("*").alias("mechanical_dnfs"))
    .orderBy("decade", F.desc("mechanical_dnfs"))
)

display(q6)

**Explanation:**

The for loop builds a single compound boolean filter by iteratively OR-ing F.lower(F.col("status")).contains(kw) for each keyword — equivalent to SQL WHERE LOWER(status) LIKE '%engine%' OR LOWER(status) LIKE '%gearbox%' …. Using F.lower first makes the match case-insensitive. F.lit(False) is a safe starting value for the OR chain (ORing anything with False returns the thing itself). F.floor(F.col("year") / 10) * 10 truncates a year to its decade — e.g. 1994 → 190 * 10 = 1990. The groupBy("name", "decade") then counts failures per team per decade. F.count("*") counts every row in the group (one row = one mechanical retirement). The result lets you compare reliability across eras — you would expect turbo-era teams from the 1980s to show high engine failure counts, while modern hybrid-era teams are far more reliable due to FIA regulations capping engine use per season.